# CNN、MNIST 与错误样本分析

## 学习目标

能够追踪卷积特征图形状，训练真实 MNIST，并分析预测错误。


## 概念模型与执行路径

卷积通过局部连接和权重共享提取空间模式，池化缩小空间尺寸，分类头把特征映射为 logits。交叉熵直接接收 logits。


### 实验 1：定位课程 CNN 组件

**实验目的**：从不同的 Jupyter 启动目录定位包含 `common` 的课程根目录，使后续能够导入共享模型、数据加载器和 checkpoint 工具。

代码检查当前目录、父目录以及当前目录下的 `07-deep-learning/pytorch`，选择第一个包含 `common` 子目录的路径。若它尚未出现在 `sys.path`，就插入最前面。重复运行不会重复添加路径。

`Path.cwd()` 表示 Jupyter 进程的工作目录，不一定是 notebook 文件所在目录。如果三个候选项都不匹配，`next(...)` 会抛出 `StopIteration`；此时应检查启动位置或项目结构。正式项目通常应安装包或从固定入口启动，而不是依赖动态修改模块搜索路径。

**预期现象**：打印路径应指向 `07-deep-learning/pytorch`，其中包含 `common/models.py` 和 `examples/train_image_classifier.py`。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：验证 CNN 的端到端张量形状

**实验目的**：用一批模拟 MNIST 图像检查特征提取器和分类头的输入输出契约。PyTorch 图像采用 NCHW：`x` 的 `(8, 1, 28, 28)` 分别表示 batch、通道、高和宽。MNIST 是灰度图，因此 `channels=1`。

`ImageClassifier` 的特征路径是 `Conv2d(1,16,3,padding=1) → ReLU → MaxPool2d(2) → Conv2d(16,32,3,padding=1) → ReLU → AdaptiveAvgPool2d((4,4))`。因此 `model.features(x)` 应输出 `(8, 32, 4, 4)`。分类头将每个样本展平为 $32\times4\times4=512$ 个特征，再由 `Linear(512,10)` 输出 `(8, 10)` logits。

logits 是每个样本对数字 0–9 的未归一化分数，训练时应直接传给 `CrossEntropyLoss`，不需要提前 softmax。预测类别可用 `logits.argmax(dim=1)` 得到。

**观察重点**：本实验只做随机输入的前向检查，logits 没有分类意义；它验证的是模型结构和 shape，而不是准确率。

In [ ]:
import torch
from common.models import ImageClassifier
model = ImageClassifier(channels=1)
x = torch.randn(8, 1, 28, 28)
features = model.features(x)
logits = model(x)
print("input:", x.shape, "features:", features.shape, "logits:", logits.shape)


### 实验 3：逐层追踪卷积特征图

**实验目的**：把实验 2 的特征提取器拆开执行，观察通道数和空间尺寸在哪一层变化。循环覆盖 `model.features` 的所有直接子层，并把上一层输出作为下一层输入。

预期 shape 路径为：

- Conv2d：`(8, 1, 28, 28) → (8, 16, 28, 28)`，padding=1 抵消 3×3 卷积核造成的边界缩小；
- ReLU：shape 不变，只把负激活截为 0；
- MaxPool2d(2)：`28×28 → 14×14`；
- 第二个 Conv2d：通道 `16 → 32`，空间仍为 `14×14`；
- 第二个 ReLU：shape 不变；
- AdaptiveAvgPool2d：把每个通道的 `14×14` 自适应汇聚为固定 `4×4`。

普通池化按固定 kernel/stride 缩放，AdaptiveAvgPool 则根据当前输入尺寸计算汇聚区域，使分类头始终收到 512 个特征。这就是模型能接受其他合理空间尺寸的原因，但输入仍必须有正确的 batch 和 channel 维。

**执行依赖**：该单元会原地把变量 `x` 改成最终特征图 `(8,32,4,4)`；若要重跑实验 2，应重新执行创建原始 `x` 的单元。

In [ ]:
for name, layer in model.features.named_children():
    x = layer(x)
    print(name, layer.__class__.__name__, tuple(x.shape))


### 实验 4：从终端训练真实 MNIST 分类器

**实验目的**：运行课程的完整图像分类训练入口，而不仅是前向 shape 检查。本单元展示命令并打印脚本路径，不会在 notebook 内自动启动长时间训练。

命令使用 quick 模式：从 MNIST 官方训练部分确定性选择 1024 个样本，其中 820 个训练、204 个验证；测试集限制为 256 个样本。训练路径加入 `RandomAffine`，验证和测试只做 `ToTensor` 与 MNIST 均值/标准差归一化。训练 loader shuffle，验证和测试不 shuffle。

脚本使用 `ImageClassifier`、交叉熵、AdamW 和基于验证准确率的 `ReduceLROnPlateau`。每轮依次训练、验证、调整学习率并更新 EarlyStopping；只有验证准确率严格改善时才覆盖 `artifacts/image_classifier.pt`。训练结束后重新加载这个最佳 checkpoint，再计算测试指标，因此测试结果对应最佳验证模型，而不是最后一轮模型。

**运行提示**：首次运行需要下载 MNIST；若已有课程 data 缓存则直接复用。quick 模式适合验证流程，不代表完整数据集上的最终准确率。可用 `--device cpu|cuda|mps` 和 `--output-dir` 控制设备与产物位置。

In [ ]:
# 首次运行会下载 MNIST；快速模式只使用小子集。
# python 07-deep-learning/pytorch/examples/train_image_classifier.py --dataset mnist --quick --epochs 3
print("完整训练入口：", PYTORCH_ROOT / "examples/train_image_classifier.py")


### 实验 5：确认训练产物并准备错误样本分析

**实验目的**：确认默认最佳 checkpoint 是否存在，为后续加载模型和收集误分类样本做准备。当前代码只打印布尔值，不会加载 checkpoint，也不会自动绘制错误样本。未运行实验 4 的终端训练时，结果通常是 `False`。

若文件存在，完整分析流程应当是：创建相同结构的 `ImageClassifier(channels=1)` → 用 `load_checkpoint` 恢复最佳权重 → `model.eval()` 并在 `torch.inference_mode()` 下遍历测试 loader → 用 `argmax(dim=1)` 取得预测 → 保存 `prediction != label` 的图像、真值、预测和置信度。

错误分析不能只挑几个显眼案例。应先构建真实类别×预测类别的混淆计数，再查看最常见错误对，例如模型把哪些数字彼此混淆；同时区分高置信错误、低置信错误、书写歧义和可能的数据问题。MNIST 输入已经标准化，绘图前若希望恢复原灰度范围，可用 $image\times0.3081+0.1307$ 反标准化并裁剪到 `[0,1]`。

**数据边界**：错误样本分析可以用于理解模型，但如果反复据此调整模型，测试集实际上就参与了开发。迭代诊断优先使用验证集，把测试集保留给最终报告。

In [ ]:
# 训练后可加载最佳检查点并收集错误样本：
from common.checkpoint import load_checkpoint
checkpoint = PYTORCH_ROOT / "artifacts/image_classifier.pt"
print("checkpoint exists:", checkpoint.exists())


## 底层机制

卷积利用局部连接和权重共享：同一个 3×3 kernel 滑过整张图，检测各位置上的相似模式。权重形状是 `(out_channels, in_channels, kernel_h, kernel_w)`，第一层因此为 `(16,1,3,3)`，参数量为 $16\times1\times3\times3+16=160$；第二层为 `(32,16,3,3)`，参数量为 4640。分类层参数量为 $512\times10+10=5130$，整网共 9930 个可训练参数。

标准卷积空间输出满足 $H_{out}=\lfloor(H+2P-D(K-1)-1)/S+1\rfloor$。本模型卷积使用 kernel=3、stride=1、padding=1，因此保持高宽；MaxPool2d(2) 才把 28 缩到 14。模型并没有第二次 2 倍 MaxPool：最终 `4×4` 来自 AdaptiveAvgPool，而不是从 14 经过两次减半得到。

AdaptiveAvgPool 固定分类头输入尺寸，降低了对原始高宽的耦合，但不代表任意尺寸都合理：过小图像可能无法形成有意义的空间特征，不同缩放还会改变数字笔画的视觉尺度。卷积的平移等变性和池化带来的局部稳健性也不是完全平移不变，边界、步幅和数据增强都会影响实际表现。

## 官方教程补充

**对应官方源文件：** `beginner_source/blitz/cifar10_tutorial.py`、`beginner_source/nn_tutorial.py`、`recipes_source/recipes/reasoning_about_shapes.py`

官方 CNN 教程要求持续追踪 NCHW：卷积改变通道和空间尺寸，池化下采样，flatten 之后才进入线性层。分类模型输出 logits，训练后不仅看 accuracy，也要按类别与错误样本检查。若替换输入分辨率或卷积配置，先用一个假 batch 验证每层 shape，避免把展平尺寸硬编码错。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

不运行代码，先回答再验证：

1. 写出从 `(8,1,28,28)` 到 `(8,32,4,4)` 的逐层 shape。
2. 本模型是否真的进行了两次 2 倍池化？为什么最终不是 `7×7`？
3. 第一、第二卷积层及分类层分别有多少参数？总参数量是多少？
4. 为什么分类头输入固定为 512？输入换成 `32×32` 时 logits shape 是什么？
5. 为什么交叉熵直接接收 logits？预测类别又为什么可以直接对 logits 做 argmax？
6. 训练脚本在哪个时机保存 checkpoint，测试前为何重新加载？
7. 若多次根据测试集错误样本改模型，会破坏什么评估边界？

## 试一试

1. **验证 shape 泛化**：分别输入 `32×32` 和 `40×24` 图像，记录逐层 shape，确认最终 logits 仍为 `(batch,10)`。
2. **核验参数量**：手算 9930 个参数，再用 `sum(p.numel() for p in model.parameters())` 验证。
3. **训练 quick 模型**：使用独立临时 `--output-dir` 训练 3 轮，检查 best checkpoint 的 epoch 和 validation accuracy metadata。
4. **系统分析错误**：加载最佳模型，在验证集收集预测，绘制混淆矩阵；选择最常见的混淆方向，并展示至少 8 个样本及真值、预测和置信度。
5. **比较增强**：保持数据划分、种子和超参数一致，分别用/不用 RandomAffine 训练，比较训练准确率与验证准确率，不要只比较单次随机结果。
6. **检查感受野**：计算最终特征单元相对输入的理论感受野，思考仅一次 MaxPool 对局部和全局信息的权衡。

## 常见错误与调试

- **输入缺少 channel 维**：MNIST 单张图常是 `(28,28)`，Conv2d 训练输入应为 `(N,1,28,28)`。在 Dataset/transform 边界检查 shape。
- **通道数不匹配**：灰度模型接收 1 通道，CIFAR-10 模型接收 3 通道。用 loader 返回的 `channels` 构造模型。
- **Flatten 尺寸错误**：修改特征层后仍硬编码旧的 Linear 输入，导致矩阵乘法失败。逐层打印 shape，或用 adaptive pooling/lazy layer 管理边界。
- **交叉熵前手动 softmax**：破坏输入语义并降低数值稳定性。训练传原始 logits。
- **验证时仍处于训练模式或记录梯度**：Dropout/BatchNorm 行为错误且浪费内存。使用 `eval()` 加 `inference_mode()`。
- **测试前未重新加载最佳 checkpoint**：最后一轮可能已过拟合。显式恢复验证指标最佳的权重。
- **训练增强污染验证集**：指标随机波动且偏离真实评估。训练和验证使用不同 Dataset 实例及 transform。
- **只看总体 accuracy**：会掩盖特定数字对、类别不平衡和高置信错误。增加混淆矩阵与分组错误分析。
- **忘记反标准化就绘图**：显示对比度异常。按训练时的 mean/std 逆变换后再展示。
- **在循环中长期保存带图 logits**：导致计算图被引用、内存增长。错误分析在 inference mode 中进行，或保存 `detach().cpu()` 的结果。